# Практика 4. Развёртывание модели качества вина (Wine Quality, UCI)

**Группа 9.** Навасардян Алик, Родина Алина, Банников Глеб, Вострова Мария.  
**Тема:** система оценки качества вина по лабораторным признакам.  
**Связь с практикой 3:** класс 1 означает `quality ≥ 7`; логистическая регрессия; порог `0.65`.

**Важно:** в переданных материалах нет полного CSV Wine Quality, поэтому этот блокнот в текущем виде показывает фактически проверенную *офлайн-демонстрацию на 69 строках настоящего UCI*. Для итоговой модели выполните `python train.py` при наличии интернета или исходных CSV, после чего обновите приведённые ниже проверки. Демо-метрики не эквивалентны метрикам из практики 3.

## 1. Задание

Реализовать обучение модели одной командой, сохранить `Pipeline` + `meta.json` + 5 эталонных прогнозов, реализовать API `/health`, `/predict`, `/model-info`, обработку ошибок `422`, автоматические тесты, Dockerfile и сценарий отката. Код по структуре адаптирован из примера с оттоком клиентов на задачи проекта Wine Quality.

In [1]:
import json, sys, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display
ROOT=Path.cwd()
print("Корень проекта:", ROOT.name)
print("CSV UCI (красное):", (ROOT/'data'/'winequality-red.csv').exists())
print("CSV UCI (белое):", (ROOT/'data'/'winequality-white.csv').exists())
print("Демо-файл:", (ROOT/'data'/'demo_wine.csv').exists())

Корень проекта: Wine_Quality_Практика_4
CSV UCI (красное): False
CSV UCI (белое): False
Демо-файл: True


## 2. Обучение и артефакты

Чтобы обучить **реальную итоговую модель** на 6 497 записях, выполните в терминале: `python train.py`. В офлайн-демонстрации мы явно указываем `--demo`; программа никак не выдаёт демо за полноценную модель.

In [2]:
result=subprocess.run([sys.executable, 'train.py', '--demo'], capture_output=True, text=True)
print(result.stdout)
assert result.returncode == 0, result.stderr

{
  "model_version": "wine-logreg-1.0.0",
  "demo_mode": true,
  "data_counts": {
    "raw_rows": 69,
    "duplicates_removed": 0,
    "unique_rows": 69,
    "train_rows": 55,
    "validation_rows": 14,
    "positive_train": 18,
    "positive_validation": 4
  },
  "metrics": {
    "precision": 1.0,
    "recall": 0.5,
    "f1": 0.6667,
    "roc_auc": 0.75,
    "pr_auc": 0.7049,
    "selected_for_tasting": 2,
    "true_positives": 2,
    "false_positives": 0,
    "false_negatives": 2,
    "true_negatives": 10
  }
}



In [3]:
meta=json.loads((ROOT/'artifacts'/'meta.json').read_text(encoding='utf-8'))
display(pd.DataFrame(meta['metrics'], index=['Значение']).T)
print('Версия:',meta['model_version'])
print('Демо режим:',meta['demo_mode'])
print('Записей:',meta['data_counts'])

,Значение
precision,1.0000
recall,0.5000
f1,0.6667
roc_auc,0.7500
pr_auc,0.7049
selected_for_tasting,2.0000
true_positives,2.0000
false_positives,0.0000
false_negatives,2.0000
true_negatives,10.0000


Версия: wine-logreg-1.0.0
Демо режим: True
Записей: {'raw_rows': 69, 'duplicates_removed': 0, 'unique_rows': 69, 'train_rows': 55, 'validation_rows': 14, 'positive_train': 18, 'positive_validation': 4}


## 3. Эталонные предсказания — проверка сохранённого Pipeline

In [4]:
import joblib
model=joblib.load(ROOT/'artifacts'/'model.joblib')
etalon=json.loads((ROOT/'artifacts'/'etalon.json').read_text(encoding='utf-8'))
print('Эталонных случаев:',len(etalon))
for i,s in enumerate(etalon,1):
    prob=float(model.predict_proba(pd.DataFrame([s['features']]))[0,1])
    assert abs(prob-s['probability']) < 1e-7
    print(f'{i}. Вероятность хорошего вина: {prob:.6f}, класс: {s["predicted_class"]}')

Эталонных случаев: 5
1. Вероятность хорошего вина: 0.504527, класс: 0
2. Вероятность хорошего вина: 0.393594, класс: 0
3. Вероятность хорошего вина: 0.805603, класс: 1
4. Вероятность хорошего вина: 0.090691, класс: 0
5. Вероятность хорошего вина: 0.103927, класс: 0


## 4. API — запросы через FastAPI TestClient (без отдельного сервера)

Статусы: `200` для валидного прогноза, `422` для некорректного запроса и `503` при недоступных артефактах. Поля запроса совпадают с UCI (включая пробелы).

In [5]:
import os
from fastapi.testclient import TestClient
from app.main import app
os.environ['WINE_ARTIFACTS_DIR']=str(ROOT/'artifacts')
with TestClient(app) as client:
    health=client.get('/health')
    valid=client.post('/predict',json=etalon[0]['features'])
    bad=client.post('/predict',json={**etalon[0]['features'],'тип_вина':'розовое'})
    info=client.get('/model-info')
    print('GET /health ->',health.status_code,health.json())
    print('POST /predict ->',valid.status_code,valid.json())
    print('POST /predict [невалидный] ->',bad.status_code)
    print('GET /model-info ->',info.status_code, info.json()['model_version'])
    assert [health.status_code,valid.status_code,bad.status_code,info.status_code] == [200,200,422,200]

GET /health -> 200 {'status': 'ok', 'model_loaded': True, 'model_version': 'wine-logreg-1.0.0'}
POST /predict -> 200 {'predicted_class': 0, 'label': 'остальное', 'probability': 0.504527, 'threshold': 0.65, 'model_version': 'wine-logreg-1.0.0'}
POST /predict [невалидный] -> 422
GET /model-info -> 200 wine-logreg-1.0.0


## 5. Автоматическое тестирование

In [6]:
result=subprocess.run([sys.executable,'-m','pytest','-q'],capture_output=True,text=True)
print(result.stdout)
assert result.returncode == 0, result.stderr

......................                                                   [100%]
22 passed in 0.39s



## 6. Контейнер и откат

**Сборка production-образа выполняется только после полного `python train.py`:**

```bash
docker build -t wine-quality-service:1.0.0 .
docker run --rm -p 8077:8077 wine-quality-service:1.0.0
curl http://127.0.0.1:8077/health
```

**Откат локальной модели (проверен автоматическим тестом):**

```bash
python train.py --version wine-logreg-1.0.0
python train.py --offline --version wine-logreg-1.1.0 --save-previous
python scripts/rollback_model.py --version wine-logreg-1.0.0
```

**Docker-сценарий:** `bash scripts/rollback.sh`. Он запускает версии 1.1.0 и 1.0.0 и сравнивает `/health`. Фактическая сборка Docker не была проверена в среде подготовки, где Docker отсутствует.

## 7. Вывод

На уровне Python реализованы обучение, валидация, артефакты, API с кодами `200/422/503`, тесты и откат модели. Проект подготовлен для загрузки в Git. Для полного завершения пунктов об обучении на UCI и Docker необходимо запустить приведённые команды в среде, где доступны оригинальные CSV и Docker. Демонстрационную модель нельзя использовать как итоговую для оценки точности.